<a href="https://colab.research.google.com/github/AnthonyMath1022/AnthonyMath1022/blob/main/Favorita_TFT(Google%20Research).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
MINICONDA_INSTALLER_SCRIPT=Miniconda3-py37_4.12.0-Linux-x86_64.sh
MINICONDA_PREFIX=/usr/local
wget -q https://repo.anaconda.com/miniconda/$MINICONDA_INSTALLER_SCRIPT
chmod +x $MINICONDA_INSTALLER_SCRIPT
./$MINICONDA_INSTALLER_SCRIPT -b -f -p $MINICONDA_PREFIX
conda install --channel defaults conda python=3.7 --yes

PREFIX=/usr/local
Unpacking payload ...
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py37h27cfd23_1003
    - ca-certificates==2022.3.29=h06a4308_1
    - certifi==2021.10.8=py37h06a4308_2
    - cffi==1.15.0=py37hd667e15_1
    - charset-normalizer==2.0.4=pyhd3eb1b0_0
    - colorama==0.4.4=pyhd3eb1b0_0
    - conda-content-trust==0.1.1=pyhd3eb1b0_0
    - conda-package-handling==1.8.1=py37h7f8727e_0
    - conda==4.12.0=py37h06a4308_0
    - cryptography==36.0.0=py37h9ce1e76_0
    - idna==3.3=pyhd3eb1b0_0
    - ld_impl_linux-64==2.35.1=h7274673_9
    - libffi==3.3=he6710b0_2
    - libgcc-ng==9.3.0=h5101ec6_17
    - libgomp==9.3.0=h5101ec6_17
    - libstdcxx-ng==9.3.0=hd4cf53a_17
    - ncurses==6.3=h7f8727e_2
    - openssl==1.1.1n=h7f8727e_0
    - pip==21.2.2=py37h06a4308_0
    - pycosat==0.6.3=py37h27cfd23_0
    - pycparser==2.21=pyhd3

                                                                                             

==> WARNING: A newer version of conda exists. <==
  current version: 4.12.0
  latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [2]:
!/usr/local/bin/python -m pip install protobuf==3.20.0 tensorflow==1.15.2
!/usr/local/bin/python -m pip install wget pyunpack patool pandas==1.0.3 scikit-learn==0.22.2.post1 kaggle

In [8]:
!/usr/local/bin/python -m pip install numpy==1.19.5

     |████████████████████████████████| 14.8 MB 12.1 MB/s 
  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.6
    Uninstalling numpy-1.21.6:
      Successfully uninstalled numpy-1.21.6


In [3]:
!git clone https://github.com/google-research/google-research.git
%cd google-research/tft

fatal: destination path 'google-research' already exists and is not an empty directory.
/content/google-research/tft


In [4]:
from google.colab import files
import os

files.upload()  # Upload your kaggle.json here

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Create the specific target directory for the official script
!mkdir -p favorita_output/data/
!kaggle competitions download favorita-grocery-sales-forecasting -p favorita_output/data/

Saving kaggle.json to kaggle (2).json
favorita-grocery-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
!unzip -n favorita_output/data/favorita-grocery-sales-forecasting.zip -d favorita_output/data/
!ls -lh favorita_output/data/

Archive:  favorita_output/data/favorita-grocery-sales-forecasting.zip
total 916M
drwxr-xr-x 2 root root 4.0K Mar 30 13:30 favorita
-rw-r--r-- 1 root root 458M Dec 11  2019 favorita-grocery-sales-forecasting.zip
-rw-r--r-- 1 root root 1.9K Dec 11  2019 holidays_events.csv.7z
-rw-r--r-- 1 root root  14K Dec 11  2019 items.csv.7z
-rw-r--r-- 1 root root 3.7K Dec 11  2019 oil.csv.7z
-rw-r--r-- 1 root root 651K Dec 11  2019 sample_submission.csv.7z
-rw-r--r-- 1 root root  648 Dec 11  2019 stores.csv.7z
-rw-r--r-- 1 root root 4.7M Dec 11  2019 test.csv.7z
-rw-r--r-- 1 root root 453M Dec 11  2019 train.csv.7z
-rw-r--r-- 1 root root 215K Dec 11  2019 transactions.csv.7z


In [6]:
import os

# 1. Fix the Pandas KeyError in script_download_data.py
download_script = 'script_download_data.py'
if os.path.exists(download_script):
    with open(download_script, 'r') as f:
        content = f.read()

    # Replace the deprecated .loc with .reindex
    content = content.replace("oil.loc[dates]", "oil.reindex(dates)")

    with open(download_script, 'w') as f:
        f.write(content)
    print(f"Fixed KeyError in {download_script}")

# 2. Fix the IndentationError in script_train_fixed_params.py
train_script = 'script_train_fixed_params.py'
if os.path.exists(train_script):
    with open(train_script, 'r') as f:
        lines = f.readlines()

    # Line 225 is at index 224. Let's fix its indentation to match standard 2 spaces.
    # We'll search around that line to be safe.
    for i in range(220, min(230, len(lines))):
        if "tf.keras.backend.set_session(default_keras_session)" in lines[i]:
            # Strip existing whitespace and add exactly 2 spaces (standard for this repo)
            lines[i] = "  " + lines[i].lstrip()

    with open(train_script, 'w') as f:
        f.writelines(lines)
    print(f"Fixed IndentationError in {train_script}")

Fixed KeyError in script_download_data.py
Fixed IndentationError in script_train_fixed_params.py


In [9]:
# Format the data using the legacy Python environment
!/usr/local/bin/python script_download_data.py favorita favorita_output

# Train the model using the legacy Python environment
!/usr/local/bin/python script_train_fixed_params.py favorita favorita_output no

Streaming output truncated to the last 5000 lines.
Getting locations for 9_1467082
Getting locations for 9_1472453
Getting locations for 9_1473393
Getting locations for 9_1473394
Getting locations for 9_1473396
Getting locations for 9_1473403
Getting locations for 9_1473405
Getting locations for 9_1473409
Getting locations for 9_1473410
Getting locations for 9_1473411
Getting locations for 9_1473412
Getting locations for 9_1473413
Getting locations for 9_1473414
Getting locations for 9_1473425
Getting locations for 9_1473427
Getting locations for 9_1473474
Getting locations for 9_1473475
Getting locations for 9_1473478
Getting locations for 9_1473480
Getting locations for 9_1473481
Getting locations for 9_1473482
Getting locations for 9_1473483
Getting locations for 9_1473484
Getting locations for 9_1473485
Getting locations for 9_1473486
Getting locations for 9_1473487
Getting locations for 9_1473499
Getting locations for 9_1473500
Getting locations for 9_1473509
Getting locations for